<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>Proyecto End-to-End y Despliegue (FastAPI / Gradio)</h2>
</div>

---
## Configuracion del entorno

> Este proyecto usa un clasificador **ligero y autocontenido** (`scikit-learn`), ideal para
> desplegar sin descargas pesadas. En la seccion 7 se indica como sustituirlo por el modelo
> Transformer afinado en el notebook 3.

In [ ]:
import random, os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.makedirs('img', exist_ok=True)
sns.set_theme(style='whitegrid')

import sklearn, fastapi, gradio
print('scikit-learn:', sklearn.__version__)
print('fastapi     :', fastapi.__version__)
print('gradio      :', gradio.__version__)


<a id="sec1"></a>
## 1. Anatomia de un pipeline end-to-end

```
   +---------+   +--------------+   +-----------+   +--------------+   +-----------+
   |  Texto  |-->| Preprocesado |-->|  Modelo   |-->|  Resultado   |-->|  Servicio |
   | crudo   |   | (limpieza)   |   | (predice) |   | (etiqueta)   |   | API / UI  |
   +---------+   +--------------+   +-----------+   +--------------+   +-----------+
```

Un buen pipeline es **reproducible**, **encapsulado** y **desplegable**. Construiremos cada pieza
y la expondremos via **Gradio** (interfaz web) y **FastAPI** (API REST).

<a id="sec2"></a>
## 2. Datos y preprocesamiento

Reutilizamos un corpus de sentimientos en espanol (embebido) y una funcion de **normalizacion**
(minusculas, sin acentos, sin puntuacion).

In [ ]:
import unicodedata

def normalizar(texto):
    texto = texto.lower()                                   # minusculas
    # quitar acentos (normalizacion unicode NFKD)
    texto = ''.join(c for c in unicodedata.normalize('NFKD', texto)
                    if not unicodedata.combining(c))
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)              # quitar puntuacion/simbolos
    texto = re.sub(r'\s+', ' ', texto).strip()              # espacios multiples
    return texto

print(normalizar('Me ENCANTO la pelicula, una obra maestra!! :)'))


In [ ]:
positivas = [
    "Me encanto la pelicula, una obra maestra.",
    "Excelente servicio y atencion al cliente.",
    "El producto supero mis expectativas, lo recomiendo.",
    "Una experiencia maravillosa, volveria sin dudarlo.",
    "La comida estaba deliciosa y el ambiente acogedor.",
    "Increible calidad por el precio, muy satisfecho.",
    "El curso fue claro, util y muy bien explicado.",
    "Fantastico trabajo, todo salio perfecto.",
    "Me siento feliz con mi compra, llego rapidisimo.",
    "Los actores brillan y la historia emociona.",
    "Un hotel estupendo con vistas espectaculares.",
    "El libro es apasionante de principio a fin.",
    "Gran atencion, personal amable y profesional.",
    "Resultados excelentes, lo usare de nuevo.",
    "Que maravilla de lugar, totalmente recomendable.",
    "Servicio rapido y muy buena calidad.",
]
negativas = [
    "Pesima experiencia, no lo recomiendo para nada.",
    "El producto llego roto y de mala calidad.",
    "Servicio lento y personal grosero.",
    "La pelicula fue aburrida y predecible.",
    "Una perdida de tiempo y de dinero.",
    "La comida estaba fria y sin sabor.",
    "Terrible atencion, nunca volvere.",
    "El hotel estaba sucio y ruidoso.",
    "Decepcionante, esperaba mucho mas.",
    "El curso fue confuso y mal organizado.",
    "Horrible, el peor servicio que he recibido.",
    "No funciona como prometen, muy frustrante.",
    "Mala calidad y precio excesivo.",
    "El libro es tedioso y dificil de seguir.",
    "Lamentable, todo salio mal desde el inicio.",
    "Atencion deficiente y mucho retraso.",
]

df = pd.DataFrame({
    'text': positivas + negativas,
    'label': [1]*len(positivas) + [0]*len(negativas)
})
df['text_norm'] = df['text'].apply(normalizar)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(df.head())
print('\\nTotal ejemplos:', len(df))


In [ ]:
# Visualizacion 1: longitud (en palabras) por clase
df['n_palabras'] = df['text_norm'].str.split().apply(len)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x='label', y='n_palabras', palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_xticklabels(['Negativo', 'Positivo'])
ax.set_title('Distribucion de longitud de texto por clase')
ax.set_xlabel('Clase'); ax.set_ylabel('# palabras')
plt.tight_layout()
plt.savefig('img/nb4_longitud_clase.png', dpi=120)
plt.show()
print('Figura guardada en img/nb4_longitud_clase.png')


<a id="sec3"></a>
## 3. Entrenamiento del modelo

Usamos un pipeline `TF-IDF + Regresion Logistica` (rapido, sin descargas). Encapsulamos
vectorizacion y clasificacion en un solo objeto de `scikit-learn`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df['text_norm'], df['label'], test_size=0.25, random_state=SEED, stratify=df['label'])

modelo = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
])
modelo.fit(X_train, y_train)
print('Modelo entrenado. Accuracy train: %.3f' % modelo.score(X_train, y_train))


<a id="sec4"></a>
## 4. Encapsulado: clase `PipelineNLP`

Encapsulamos limpieza + prediccion para reutilizar el mismo objeto en Gradio y FastAPI.

In [ ]:
class PipelineNLP:
    # Pipeline NLP reutilizable: normaliza el texto y predice el sentimiento.
    ETIQUETAS = {0: 'NEGATIVO', 1: 'POSITIVO'}

    def __init__(self, modelo):
        self.modelo = modelo

    def _limpiar(self, texto):
        return normalizar(texto)

    def predecir(self, texto):
        limpio = self._limpiar(texto)
        pred = int(self.modelo.predict([limpio])[0])
        proba = float(self.modelo.predict_proba([limpio])[0][pred])
        return {'etiqueta': self.ETIQUETAS[pred], 'confianza': round(proba, 4),
                'texto_normalizado': limpio}

pipeline_nlp = PipelineNLP(modelo)
print(pipeline_nlp.predecir('El servicio fue excelente y muy rapido'))
print(pipeline_nlp.predecir('Una experiencia horrible, no vuelvo'))

<a id="sec5"></a>
## 5. Evaluacion final

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['NEGATIVO', 'POSITIVO'], zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['NEGATIVO', 'POSITIVO'], yticklabels=['NEGATIVO', 'POSITIVO'], ax=ax)
ax.set_title('Matriz de confusion del proyecto final')
ax.set_xlabel('Prediccion'); ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig('img/nb4_matriz_confusion.png', dpi=120)
plt.show()
print('Figura guardada en img/nb4_matriz_confusion.png')


In [ ]:
# Visualizacion 3: confianza del modelo en frases nuevas
frases_demo = [
    "Me encanta este lugar, todo perfecto",
    "Pesimo, no lo recomiendo",
    "El producto es aceptable, cumple",
    "Servicio lento y caro",
]
res = [pipeline_nlp.predecir(f) for f in frases_demo]
colores = ['#2ecc71' if r['etiqueta'] == 'POSITIVO' else '#e74c3c' for r in res]
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x=[r['confianza'] for r in res], y=frases_demo, palette=colores, ax=ax)
for i, r in enumerate(res):
    ax.text(r['confianza'] + 0.01, i, r['etiqueta'], va='center', fontsize=9)
ax.set_xlim(0, 1.15)
ax.set_title('Confianza de prediccion en frases nuevas')
ax.set_xlabel('Confianza')
plt.tight_layout()
plt.savefig('img/nb4_confianza.png', dpi=120)
plt.show()
print('Figura guardada en img/nb4_confianza.png')


<a id="sec6"></a>
## 6. Despliegue con Gradio

Gradio crea una **interfaz web** en pocas lineas. La funcion de prediccion recibe texto y
devuelve la etiqueta y la confianza.

> **Nota:** `demo.launch()` abre un servidor local. Detén la celda (boton stop) para liberarlo.
> Usa `share=True` para obtener un enlace publico temporal.

In [ ]:
import gradio as gr

def predecir_gradio(texto):
    if not texto.strip():
        return {'NEGATIVO': 0.0, 'POSITIVO': 0.0}
    limpio = normalizar(texto)
    probas = modelo.predict_proba([limpio])[0]
    return {'NEGATIVO': float(probas[0]), 'POSITIVO': float(probas[1])}

demo = gr.Interface(
    fn=predecir_gradio,
    inputs=gr.Textbox(lines=3, label='Escribe una resena en espanol'),
    outputs=gr.Label(num_top_classes=2, label='Sentimiento'),
    title='Analizador de Sentimientos NLP 2026',
    description='Proyecto final - Modulo 3. Clasifica resenas como positivas o negativas.',
    examples=[['Me encanto, totalmente recomendable'],
              ['Una perdida de tiempo, pesimo servicio']],
)

# Descomenta para lanzar la interfaz (bloquea la celda hasta detenerla):
# demo.launch()
print('Interfaz Gradio definida. Descomenta demo.launch() para ejecutarla.')


<a id="sec7"></a>
## 7. Despliegue con FastAPI

FastAPI expone el modelo como **API REST**. Escribimos la app en `app_api.py` y la probamos
con `TestClient` (sin necesidad de levantar un servidor).

Para ejecutarla como servidor real:
```bash
uvicorn app_api:app --reload
```
Luego visita `http://127.0.0.1:8000/docs` para la documentacion interactiva.

> Para usar el **Transformer afinado** del notebook 3 en lugar del modelo sklearn, guarda alli el
> modelo con `trainer.save_model('modelo_beto')` y cargalo aqui con
> `pipeline('text-classification', model='modelo_beto')`.

In [ ]:
codigo_api = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="API NLP 2026 - Analisis de Sentimientos")
modelo = joblib.load("modelo_sentimientos.joblib")
ETIQUETAS = {0: "NEGATIVO", 1: "POSITIVO"}

class Entrada(BaseModel):
    texto: str

@app.get("/")
def raiz():
    return {"mensaje": "API NLP 2026 activa. Usa POST /predecir"}

@app.post("/predecir")
def predecir(entrada: Entrada):
    pred = int(modelo.predict([entrada.texto])[0])
    proba = float(modelo.predict_proba([entrada.texto])[0][pred])
    return {"etiqueta": ETIQUETAS[pred], "confianza": round(proba, 4)}
'''

with open('app_api.py', 'w', encoding='utf-8') as f:
    f.write(codigo_api)

# Guardamos el modelo entrenado para que la API lo cargue
import joblib
joblib.dump(modelo, 'modelo_sentimientos.joblib')
print('Archivos creados: app_api.py y modelo_sentimientos.joblib')


In [ ]:
# Prueba de la API SIN levantar servidor, usando TestClient
from fastapi.testclient import TestClient
import importlib, app_api
importlib.reload(app_api)
client = TestClient(app_api.app)

print('GET / ->', client.get('/').json())
for frase in ['Excelente atencion, muy recomendable', 'Horrible experiencia, no vuelvo']:
    r = client.post('/predecir', json={'texto': frase})
    print(f'POST /predecir  "{frase}"  ->  {r.json()}')


In [ ]:
# === TU SOLUCION AQUI (Proyecto Final) ===

